In [1]:
# ===============================================================================================
# ===============================================================================================
#
# Grover_Simulacion.ipynb
#
# Emulador matemático de un computador cuántico a partir de una librería de funciones (puertas)
# Estructura secuencial (circuito) y formulación de vectores de estado. 
# Simulación cuántica del algoritmo de Grover a partir de oráculo y operador de dispersión 
# codificados en clave de puertas. Aplicado para la resolución de un Sudoku 2x2.
# 
#
# @author Antonio Ramírez Prieto
# 
# Simulación para TFG en el Grado en Física
#
# ===============================================================================================
# ===============================================================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib import rcParams
from collections import Counter
import os


# ============================================================
# PARTE I: ARQUITECTURA CIRCUITO / BACKEND
# ============================================================

class CircuitoCuantico:
    """Lista de puertas e instrucciones. No se realizan cálculos"""

    def __init__(self, n_qubits):
        self.n = n_qubits
        self.instrucciones = []

    def H(self, q):    self.instrucciones.append(('H', q))
    def X(self, q):    self.instrucciones.append(('X', q))
    def Z(self, q):    self.instrucciones.append(('Z', q))  # La puerta Z se definió por una función comparativa, finalmente no es necesaria.
    def CNOT(self, c, t):       self.instrucciones.append(('CNOT', c, t))
    def CCNOT(self, c1, c2, t): self.instrucciones.append(('CCNOT', c1, c2, t))
    def barrera(self, etiqueta=None):
        self.instrucciones.append(('BARRERA', etiqueta))    # La puerta barrera se define por completitud, pero no es necesaria.

    def __repr__(self):
        n_puertas = sum(1 for i in self.instrucciones if i[0] != 'BARRERA')
        return f"CircuitoCuantico(n={self.n}, puertas={n_puertas})"


# Matrices unitarias de las puertas de 1 q
_PUERTAS_1Q = {
    'H': np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2),
    'X': np.array([[0, 1], [1, 0]], dtype=complex),
    'Z': np.array([[1, 0], [0, -1]], dtype=complex),
}


def _perm_cnot(n, controles, objetivo):
    """Permutación de índices (puerta CNOT)"""
    idx = np.arange(2**n)
    mascara = np.ones(2**n, dtype=bool)
    for c in controles:
        mascara &= ((idx >> (n - 1 - c)) & 1).astype(bool)
    return np.where(mascara, idx ^ (1 << (n - 1 - objetivo)), idx)


class SimuladorVectorEstado:
    """Definición del vector de estado e instrucciones del circuito.
    (Backend de la simulación)"""

    def __init__(self):
        self.estado = None
        self.n = 0

    def ejecutar(self, circuito):
        """Inicializa en /ket{00...0}"""
        self.n = circuito.n
        N = 2**self.n
        self.estado = np.zeros(N, dtype=complex)
        self.estado[0] = 1.0
        for instr in circuito.instrucciones:    # Recorre todas las instrucciones
            self._aplicar(instr)
        return self.estado

    def _aplicar(self, instr):
        tipo = instr[0]
        if tipo in _PUERTAS_1Q:
            self._puerta_1q(_PUERTAS_1Q[tipo], instr[1])
        elif tipo == 'CNOT':
            perm = _perm_cnot(self.n, [instr[1]], instr[2])
            self.estado = self.estado[perm]
        elif tipo == 'CCNOT':
            perm = _perm_cnot(self.n, [instr[1], instr[2]], instr[3])
            self.estado = self.estado[perm]
        elif tipo == 'BARRERA':        
            pass
        else:
            raise ValueError(f"Instrucción desconocida: {tipo}")

    def _puerta_1q(self, U, qubit):
        """Toma el vect de estaod como tensor y aplica U (2x2) sobre el qubit n """
        n = self.n
        izq, der = 2**qubit, 2**(n - 1 - qubit)
        psi = self.estado.reshape(izq, 2, der)
        psi = np.tensordot(U, psi, axes=([1], [1]))
        psi = np.moveaxis(psi, 0, 1)
        self.estado = np.ascontiguousarray(psi).reshape(2**n)

    def medir(self):
        """Medida proyectiva: devuelve el índice del resultado y colapsa el estado"""
        probs = np.abs(self.estado)**2
        probs /= probs.sum()
        resultado = int(np.random.choice(len(self.estado), p=probs))
        self.estado = np.zeros(len(self.estado), dtype=complex)
        self.estado[resultado] = 1.0
        return resultado

    def probabilidades(self):
        """Probabilidad como cuadrado de la amplitud"""
        return np.abs(self.estado)**2

    def norma(self):
        """Norma de la probabilidad (debe mantenerse 1 al final de todas las ejecuciones)"""
        return float(np.real(np.dot(self.estado.conj(), self.estado)))


# ============================================================
# PARTE II: PROBLEMA CLÁSICO DEL SUDOKU 2×2
# ============================================================
#
# Cuadrícula 2x2, valores en {0,1} 
# (qubit 0, celda 1; qubit 1, celda 2).
# q0=c00, q1=c01, q2=c10, q3=c11
#
# Restricciones por fila c00 \neq c01,  c10 \neq c11 
# Restricciones por col  c00 \neq c10,  c01 \neq c11

def encontrar_soluciones():
    """Métodoo clásico: devuelve los índices {0..15} que son solución al problema"""
    soluciones = []
    
    for x in range(16):
        b = [(x >> (3 - i)) & 1 for i in range(4)]
        c00, c01, c10, c11 = b
        if c00 != c01 and c10 != c11 and c00 != c10 and c01 != c11:
            soluciones.append(x)
                             
    return soluciones






In [2]:
# ============================================================
# PARTE III: ORÁCULO, DIFUSIÓN Y CIRCUITO DE GROVER
# ============================================================
#
# Circuito de 11 qubits
#   q0..q3  como registro principal (c00,c01,c10,c11)
#   q4..q7  son ancillas XOR (una por restricción)
#   q8,q9   son ancillas AND
#   q10     es ancilla de fase (se inicializa en \ket{-})

N_TOTAL    = 11
N_PRINCIPAL = 4
REGISTRO   = (0, 1, 2, 3)
XOR_ANC    = (4, 5, 6, 7)
AND_ANC    = (8, 9)
FASE_ANC   = 10


def oraculo_sudoku(c):
    """Oráculo del problema Sudoku 2x2

    Implementa f(c00,c01,c10,c11) = (f0 XOR f1 XOR c0 XOR c1)
    
    1. XOR de cada restricción (CNOT sobre ancilla en /ket{0})
    2. AND par a par (CCNOT sobre ancilla en /ket{0})
    3. Phase kickback: fase -1 si solución
    4. Descomputación
    """
    q0, q1, q2, q3 = REGISTRO
    a_f0, a_f1, a_c0, a_c1 = XOR_ANC
    b0, b1 = AND_ANC

    # Paso 1: CNOT
    c.CNOT(q0, a_f0); c.CNOT(q1, a_f0)   # a_f0 = q0 XOR q1  (fila 0)
    c.CNOT(q2, a_f1); c.CNOT(q3, a_f1)   # a_f1 = q2 XOR q3  (fila 1)
    c.CNOT(q0, a_c0); c.CNOT(q2, a_c0)   # a_c0 = q0 XOR q2  (col 0)
    c.CNOT(q1, a_c1); c.CNOT(q3, a_c1)   # a_c1 = q1 XOR q3  (col 1)

    # Paso 2: CCNOT (AND)
    c.CCNOT(a_f0, a_f1, b0)       # b0 = a_f0 AND a_f1
    c.CCNOT(a_c0, a_c1, b1)       # b1 = a_c0 AND a_c1
    
    # Paso 3: phase kickback
    c.CCNOT(b0, b1, FASE_ANC)
                
    # Paso 4: descomputación (orden inverso)
    c.CCNOT(a_c0, a_c1, b1)
    c.CCNOT(a_f0, a_f1, b0)
    c.CNOT(q3, a_c1); c.CNOT(q1, a_c1)
    c.CNOT(q2, a_c0); c.CNOT(q0, a_c0)
    c.CNOT(q3, a_f1); c.CNOT(q2, a_f1)
    c.CNOT(q1, a_f0); c.CNOT(q0, a_f0)


def operador_difusion(c):
    """Implementa el operador de difusión del algoritmo
    Reutiliza q4,q5 como ancillas (están en /ket{0} tras el oráculo).
    """
    anc_a, anc_b = XOR_ANC[0], XOR_ANC[1]
    
    for q in REGISTRO: c.H(q)
    for q in REGISTRO: c.X(q)
        
    # Z multicontrolada por 4 qubits
    c.CCNOT(REGISTRO[0], REGISTRO[1], anc_a)
    c.CCNOT(REGISTRO[2], REGISTRO[3], anc_b)
    c.CCNOT(anc_a, anc_b, FASE_ANC)

    # Descomputación
    c.CCNOT(REGISTRO[2], REGISTRO[3], anc_b)
    c.CCNOT(REGISTRO[0], REGISTRO[1], anc_a)
    
    for q in REGISTRO: c.X(q)
    for q in REGISTRO: c.H(q)


def construir_circuito_grover(R):
    
    """Operador de Grover R iteraciones"""
    circ = CircuitoCuantico(N_TOTAL)
    
    # Ancilla de fase en \ket{-}
    circ.X(FASE_ANC); circ.H(FASE_ANC)
    
    # Superposición uniforme sobre el registro
    for q in REGISTRO: circ.H(q)
        
    for _ in range(R):
        oraculo_sudoku(circ)
        operador_difusion(circ)
        
    return circ


def prob_marginal_registro(estado):
    """Hasta aquí tengo un vector de N = 2^11 amplitudes, lo pliego para devolver
    las 2^4 que me interesan, axis=1 y limpiar las residuales"""
    
    psi = estado.reshape(2**N_PRINCIPAL, 2**(N_TOTAL - N_PRINCIPAL))
    return np.sum(np.abs(psi)**2, axis=1)


In [3]:

# ============================================================
# PARTE IV: ALGORITMO DE GROVER (ejecución)
# ============================================================
# Ejecuta el algoritmo de Grover para el Sudoku 2x2
# Empleamos el valor M obtenido por el método clásico (hardcodeamos el número de soluciones)
#

def algoritmo_grover(soluciones, verbose=True):
    
    N = 2**N_PRINCIPAL
    M = len(soluciones)
    R = max(1, int(round((np.pi / 4) * np.sqrt(N / M))))

    # Log de la simulación
    if verbose:
        print(f"\n{'='*55}")
        print(" ALGORITMO DE GROVER — Sudoku 2×2")
        print(f"{'='*55}")
        print(f"  Qubits registro : {N_PRINCIPAL}  |  Espacio : N = {N}")
        print(f"  Soluciones      : M = {M}")
        print(f"  Iteraciones     : R = {R}  (π/4·√(N/M) = {(np.pi/4)*np.sqrt(N/M):.2f})")
        print(f"  Complejidad     : clásica O({N}), cuántica O(√{N}≈{np.sqrt(N):.1f})")
        print(f"{'-'*55}")

    backend = SimuladorVectorEstado()
    historial = []

    for k in range(R + 1):
        estado_k = backend.ejecutar(construir_circuito_grover(k))
        probs_k  = prob_marginal_registro(estado_k)
        p_sol    = float(np.sum(probs_k[soluciones]))
        historial.append(p_sol)
        if verbose:
            label = "inicial" if k == 0 else f"iter {k}"
            print(f"  k={k} ({label}): P(solución) = {p_sol:.6f}  |norma| = {backend.norma():.10f}")

    # Ejecutar el circuito final para la medida
    backend.ejecutar(construir_circuito_grover(R))
    probs_finales  = prob_marginal_registro(backend.estado)

    # Ejecutamos k=0 para guardar distribución inicial
    probs_ini_dist = prob_marginal_registro(
        SimuladorVectorEstado().ejecutar(construir_circuito_grover(0)))

    medido_total = backend.medir()
    idx_registro = medido_total >> (N_TOTAL - N_PRINCIPAL)
    correcto     = idx_registro in soluciones

    if verbose:
        print(f"\n  Medida: |{format(idx_registro,'04b')}> (índice {idx_registro})"
              f" → {'✓ solución' if correcto else '✗ no es solución'}")
        print(f"{'='*55}\n")

    return {
        'resultado'       : idx_registro,
        'correcto'        : correcto,
        'probs_iniciales' : probs_ini_dist,
        'probs_finales'   : probs_finales,
        'historial'       : historial,
        'iteraciones'     : R,
    }



In [55]:

# ============================================================
# PARTE V: VISUALIZACIÓN
# ============================================================
# Apartado generado por un motor de inteligencia artificial para la representación gráfica
# de los resultados de la simulación.


# ===============================================================================================
CARPETA_OUT = os.path.join(os.path.expanduser('~'), 'Desktop', 'grover_figuras')
os.makedirs(CARPETA_OUT, exist_ok=True)

def ruta(nombre_archivo):
    """Ruta de salida para la generación de archivos"""
    return os.path.join(CARPETA_OUT, nombre_archivo)
# ===============================================================================================

def _marco_academico(ax):
    """Aplica el acabado sobrio estándar a un eje individual."""
    ax.set_facecolor(PAL['fondo_ax'])
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color('#333333')
        ax.spines[spine].set_linewidth(0.8)
    ax.tick_params(which='both', direction='out', color='#333333',
                   labelcolor='#222222', length=4, width=0.7)


def _etiqueta_panel(ax, letra):
    """Etiqueta tipo (a), (b) en esquina superior izquierda — estilo revista."""
    ax.text(-0.10, 1.04, f'({letra})', transform=ax.transAxes,
            fontsize=11, fontweight='bold', va='bottom', ha='left',
            color='#222222', fontfamily='serif')


# ============================================================
# ESTILO GLOBAL — Configura rcParams para un estilo sobrio, elegante y propio de publicación académica.
# ============================================================

def aplicar_estilo_tfm():
    rcParams.update({
        # Familia tipográfica
        'font.family'        : 'serif',
        'font.serif'         : ['DejaVu Serif', 'Times New Roman', 'Times', 'serif'],
        'mathtext.fontset'   : 'dejavuserif',

        # Tamaños
        'font.size'          : 10,
        'axes.titlesize'     : 11,
        'axes.labelsize'     : 10,
        'xtick.labelsize'    : 8.5,
        'ytick.labelsize'    : 8.5,
        'legend.fontsize'    : 8.5,
        'figure.titlesize'   : 12,

        # Fondo blanco puro (papel)
        'figure.facecolor'   : 'white',
        'axes.facecolor'     : '#fafafa',
        'savefig.facecolor'  : 'white',

        # Bordes de ejes
        'axes.edgecolor'     : '#333333',
        'axes.linewidth'     : 0.8,
        'axes.spines.top'    : False,
        'axes.spines.right'  : False,

        # Grid
        'axes.grid'          : False,

        # Ticks
        'xtick.direction'    : 'out',
        'ytick.direction'    : 'out',
        'xtick.major.width'  : 0.7,
        'ytick.major.width'  : 0.7,
        'xtick.color'        : '#333333',
        'ytick.color'        : '#333333',

        # Leyenda
        'legend.frameon'     : True,
        'legend.framealpha'  : 0.92,
        'legend.edgecolor'   : '#cccccc',

        # Figura
        'figure.dpi'         : 150,
        'savefig.dpi'        : 200,
        'savefig.bbox'       : 'tight',
        'savefig.pad_inches' : 0.08,

        # Líneas
        'lines.linewidth'    : 1.6,
        'lines.markersize'   : 6,
    })
PAL ={
    'k1_sol'    : '#e07b39',   # naranja tostado  — solución en k=1
    'k1_no_sol' : '#f2c9a8',   # naranja claro    — no solución en k=1
    'sol'      : '#1a4f8a',   # azul oscuro — estados solución
    'no_sol'   : '#9db8d2',   # azul gris claro — no solución
    'rojo'     : '#8b1a1a',   # rojo oscuro — referencias / clásico
    'verde'    : '#2d6a4f',   # verde oscuro — cuántico / éxito
    'dorado'   : '#8b6914',   # dorado oscuro — líneas auxiliares
    'gris'     : '#555555',   # gris medio
    'gris_claro': '#aaaaaa',  # gris claro
    'morado'   : '#4a2a6b',   # morado — speedup
    'fondo_ax' : '#fafafa',   # fondo de axes
}

DPI_OUT = 200

# ──────────────────────────────────────────────
# Figura 1: Distribuciones de probabilidad
# ──────────────────────────────────────────────

def graficar_distribucion_probabilidad(res, soluciones):
    """
    Figura 1.
    (a) Distribución uniforme inicial  k=0
    (b) Superposición de distribuciones k=1 (naranja, delante) y k=2 (azul, detrás)
    """
    aplicar_estilo_tfm()
    N       = 2**N_PRINCIPAL
    estados = [format(x,'04b') for x in range(N)]

    # Distribución k=1 calculada internamente
    probs_k1 = prob_marginal_registro(
        SimuladorVectorEstado().ejecutar(construir_circuito_grover(1)))

    colores_ini = [PAL['sol'] if i in soluciones else PAL['no_sol'] for i in range(N)]
    colores_k2  = [PAL['sol'] if i in soluciones else PAL['no_sol'] for i in range(N)]
    colores_k1  = [PAL['k1_sol'] if i in soluciones else PAL['k1_no_sol'] for i in range(N)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6),
                                   gridspec_kw={'wspace': 0.38})
    fig.suptitle(
        r'Distribuciones de probabilidad — Algoritmo de Grover (Sudoku $2\times2$)',
        fontsize=12, fontweight='bold', color='#1a1a1a', y=1.02)

    # ── Panel (a): distribución inicial k=0 ──
    _marco_academico(ax1)
    ax1.bar(range(N), res['probs_iniciales'], color=colores_ini,
            edgecolor='white', linewidth=0.5, width=0.72)
    ax1.axhline(1/N, color=PAL['dorado'], linestyle='--',
                linewidth=1.1, label=fr'$1/N = {1/N:.4f}$', zorder=3)
    ax1.set_title(r'Estado inicial $H^{\otimes 4}|0\rangle$  $(k=0)$',
                  fontsize=10, pad=6, color='#1a1a1a')
    ax1.set_xlabel(r'Estado $|x\rangle$', labelpad=4)
    ax1.set_ylabel('Probabilidad', labelpad=4)
    ax1.set_xticks(range(N))
    ax1.set_xticklabels(estados, rotation=45, fontsize=7.5, fontfamily='monospace')
    ax1.set_ylim(0, 0.22)
    ax1.legend(loc='upper right', fontsize=8)
    _etiqueta_panel(ax1, 'a')

    # ── Panel (b): k=2 detrás + k=1 delante ──
    _marco_academico(ax2)

    ax2.bar(range(N), res['probs_finales'], color=colores_k2,
            edgecolor='white', linewidth=0.5, width=0.72,
            alpha=0.90, zorder=2)
    ax2.bar(range(N), probs_k1, color=colores_k1,
            edgecolor='white', linewidth=0.5, width=0.44,
            alpha=1.0, zorder=3)
    ax2.axhline(1/N, color=PAL['dorado'], linestyle='--',
                linewidth=1.1, label=fr'$1/N = {1/N:.4f}$', zorder=4)

    for s in soluciones:
        ax2.annotate(f'{res["probs_finales"][s]:.3f}',
                     xy=(s, res['probs_finales'][s]),
                     xytext=(18, 10), textcoords='offset points',
                     ha='left', fontsize=7.5, color=PAL['sol'], fontweight='bold',
                     arrowprops=dict(arrowstyle='-', color=PAL['sol'], lw=0.6, alpha=0.6))
        ax2.annotate(f'{probs_k1[s]:.3f}',
                     xy=(s, probs_k1[s]),
                     xytext=(-65, 25), textcoords='offset points',
                     ha='left', fontsize=7.5, color='#a0450a', fontweight='bold',
                     arrowprops=dict(arrowstyle='-', color='#a0450a', lw=0.6, alpha=0.6))

    ax2.set_title(r'Superposición de distribuciones $k=1$ y $k=2$',
                  fontsize=10, pad=6, color='#1a1a1a')
    ax2.set_xlabel(r'Estado $|x\rangle$', labelpad=4)
    ax2.set_ylabel('Probabilidad', labelpad=4)
    ax2.set_xticks(range(N))
    ax2.set_xticklabels(estados, rotation=45, fontsize=7.5, fontfamily='monospace')
    ax2.set_ylim(0, 1.10)
    ax2.legend(loc='upper right', fontsize=8)
    _etiqueta_panel(ax2, 'b')

    leyenda_comun = [
        Patch(facecolor=PAL['sol'],       label=r'Solución ($k=2$)'),
        Patch(facecolor=PAL['no_sol'],    label=r'No solución ($k=2$)'),
        Patch(facecolor=PAL['k1_sol'],    label=r'Solución ($k=1$)'),
        Patch(facecolor=PAL['k1_no_sol'], label=r'No solución ($k=1$)'),
    ]
    fig.legend(handles=leyenda_comun, loc='lower center', ncol=4,
               fontsize=8, frameon=True, edgecolor='#cccccc',
               bbox_to_anchor=(0.5, -0.07))

    plt.savefig(ruta('fig1_distribuciones.png'), dpi=DPI_OUT, bbox_inches='tight')
    plt.close()
    print("  → fig1_distribuciones.png guardada")
# ──────────────────────────────────────────────
# Figura 2: Histograma estadístico
# ──────────────────────────────────────────────
def graficar_histograma_estadistico(medidas, soluciones, N_runs):
    aplicar_estilo_tfm()
    N         = 2**N_PRINCIPAL
    estados   = [format(x, '04b') for x in range(N)]
    conteos   = Counter(medidas)
    frecuencias = [conteos.get(x, 0) for x in range(N)]
    colores   = [PAL['sol'] if i in soluciones else PAL['no_sol'] for i in range(N)]
    n_exitos  = sum(1 for m in medidas if m in soluciones)
    M         = len(soluciones)

    fig, ax = plt.subplots(figsize=(11, 4.2))
    _marco_academico(ax)

    bars = ax.bar(range(N), frecuencias, color=colores,
                  edgecolor='white', linewidth=0.5, width=0.72)

    for i, (bar, freq) in enumerate(zip(bars, frecuencias)):
        if freq > 0:
            pct = 100 * freq / N_runs
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.35,
                    f'{freq}\n({pct:.1f}%)',
                    ha='center', va='bottom', fontsize=7,
                    color=PAL['sol'] if i in soluciones else PAL['gris'],
                    fontweight='bold' if i in soluciones else 'normal')

    # ax.axhline(N_runs / N, color=PAL['dorado'], linestyle='--', linewidth=1.1,
    #            label=fr'Uniforme: $N_\mathrm{{runs}}/N = {N_runs/N:.1f}$', zorder=3)
    p_emp = n_exitos / N_runs
    # ax.axhline(N_runs * p_emp / M, color=PAL['verde'], linestyle=':',
    #            linewidth=1.3,
    #            label=fr'$P(\mathrm{{sol.}})_\mathrm{{emp.}}/M = {N_runs*p_emp/M:.1f}$',
    #            zorder=3)

    leyenda_barras = [
        Patch(facecolor=PAL['sol'],    label=f'Solución ({n_exitos} medidas)'),
        Patch(facecolor=PAL['no_sol'], label=f'No solución ({N_runs - n_exitos} medidas)'),
    ]
    # handles_lineas, labels_lineas = ax.get_legend_handles_labels()
    ax.legend(handles=leyenda_barras,
              fontsize=8, loc='upper left', ncol=2)
    

    ax.set_title(
        fr'Histograma de resultados de medida — {N_runs} ejecuciones '
        fr'(tasa de éxito: {100*n_exitos/N_runs:.1f}%)',
        fontsize=10, pad=6)
    ax.set_xlabel(r'Estado medido $|x\rangle$', labelpad=4)
    ax.set_ylabel('Frecuencia absoluta', labelpad=4)
    ax.set_xticks(range(N))
    ax.set_xticklabels(estados, rotation=45, fontsize=7.5, fontfamily='monospace')
    ax.set_ylim(0, max(frecuencias) * 1.28)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    # _etiqueta_panel(ax, 'a')

    plt.savefig(ruta('fig2_histograma_estadistico.png'), dpi=DPI_OUT, bbox_inches='tight')
    plt.close()
    print("  → fig2_histograma_estadistico.png guardada")

# ──────────────────────────────────────────────
# Figura 3: Amplificación de amplitud
# ──────────────────────────────────────────────
def graficar_amplificacion_amplitud(res, soluciones):
    aplicar_estilo_tfm()
    N     = 2**N_PRINCIPAL
    M     = len(soluciones)
    theta = np.arcsin(np.sqrt(M / N))

    fig, ax = plt.subplots(figsize=(9, 4.2))
    _marco_academico(ax)

    # Curva analítica continua
    k_cont   = np.linspace(0, len(res['historial']) - 1, 500)
    p_anal   = np.sin((2 * k_cont + 1) * theta)**2
    ax.plot(k_cont, p_anal, color=PAL['gris_claro'], linewidth=1.4,
            linestyle='--', zorder=2,
            label=r'Analítica: $\sin^2((2k+1)\theta)$')

    # Puntos simulados
    iters = list(range(len(res['historial'])))
    ax.plot(iters, res['historial'], 'o-', color=PAL['sol'], linewidth=1.8,
            markersize=7, markerfacecolor='white', markeredgewidth=1.8,
            zorder=4, label='Simulación (vector de estado)')

    for k, p in zip(iters, res['historial']):
        ax.annotate(f'{p:.4f}', xy=(k, p), xytext=(0, 10),
                    textcoords='offset points', ha='center', fontsize=8.5,
                    color=PAL['sol'])

    # Referencias
    ax.axhline(1.0, color=PAL['gris_claro'], linestyle=':', linewidth=0.9, alpha=0.7)
    # ax.axhline(1/N, color=PAL['dorado'], linestyle='--', linewidth=1.0,
    #            alpha=0.8, label=fr'$1/N = {1/N:.4f}$ (uniforme)')
    # ax.axvline(res['iteraciones'], color=PAL['rojo'], linestyle=':',
               # linewidth=1.2, label=f'$R_{{\\mathrm{{opt}}}} = {res["iteraciones"]}$')

    ax.set_title(r'Amplificación de amplitudes: $P(\mathrm{sol.})$ vs iteraciones',
                 fontsize=10, pad=6)
    ax.set_xlabel(r'Número de iteraciones $k$', labelpad=4)
    ax.set_ylabel(r'$P_{sol}$', labelpad=4)
    ax.set_xticks(iters)
    ax.set_xticklabels([f'$k={k}$' for k in iters], fontsize=9)
    ax.set_ylim(-0.05, 1.20)
    ax.legend(loc='upper left', fontsize=8.5)
    ax.yaxis.grid(True, linestyle=':', linewidth=0.6, color='#cccccc', alpha=0.8)
    # _etiqueta_panel(ax, 'a')

    plt.savefig(ruta('fig3_amplificacion.png'), dpi=DPI_OUT, bbox_inches='tight')
    plt.close()
    print("  → fig3_amplificacion.png guardada")


# ──────────────────────────────────────────────
# Figura 4: Geometría — rotación hasta 180°
# ──────────────────────────────────────────────
def graficar_rotacion_grover(soluciones, R_max=8):
    aplicar_estilo_tfm()
    N     = 2**N_PRINCIPAL
    M     = len(soluciones)
    theta = np.arcsin(np.sqrt(M / N))
    R_opt = max(1, int(round(np.pi / (4 * theta))))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5),
                                   gridspec_kw={'wspace': 0.40})
    fig.suptitle(
        rf'Geometría del algoritmo de Grover — plano $|\alpha\rangle$-$|\beta\rangle$ '
        rf'$(N={N},\ M={M},\ \theta\approx{np.degrees(theta):.2f}^\circ)$',
        fontsize=11, fontweight='bold', color='#1a1a1a', y=1.02)

    # ── Panel izquierdo: rotaciones ──
    _marco_academico(ax1)
    cmap = plt.cm.Blues
    for k in range(R_max + 1):
        ang   = (2 * k + 1) * theta
        alpha = 0.35 + 0.65 * k / max(R_max, 1)
        color = cmap(0.35 + 0.60 * k / max(R_max, 1))
        ax1.annotate('', xy=(np.cos(ang), np.sin(ang)), xytext=(0, 0),
                     arrowprops=dict(arrowstyle='->', color=color,
                                     lw=1.6, alpha=alpha))
        r_lbl = 1.12
        ax1.text(np.cos(ang) * r_lbl, np.sin(ang) * r_lbl,
                 f'$k={k}$', fontsize=8, color=color,
                 ha='center', va='center', alpha=alpha)

    # Arco θ
    ang_arco = np.linspace(0, theta, 80)
    ax1.plot(np.cos(ang_arco) * 0.22, np.sin(ang_arco) * 0.22,
             color=PAL['dorado'], lw=1.3)
    ax1.text(0.27, 0.03, r'$\theta$', fontsize=9.5, color=PAL['dorado'])

    # Ejes de referencia |α⟩ y |β⟩
    ax1.axhline(0, color='#aaaaaa', lw=0.7, zorder=0)
    ax1.axvline(0, color='#aaaaaa', lw=0.7, zorder=0)

    ax1.set_xlim(-1.30, 1.42)
    ax1.set_ylim(-0.12, 1.42)
    ax1.set_aspect('equal')
    ax1.set_xlabel(r'$|\alpha\rangle$ (no soluciones)', labelpad=4)
    ax1.set_ylabel(r'$|\beta\rangle$ (soluciones)', labelpad=4)
    ax1.set_title(r'Rotación del estado cuántico', fontsize=10, pad=6)
    ax1.grid(False)

    # Colorbar discreta
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, R_max))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax1, shrink=0.65, pad=0.03, aspect=18)
    cbar.set_label('Iteración $k$', fontsize=8.5)
    cbar.ax.tick_params(labelsize=7.5)
    _etiqueta_panel(ax1, 'a')

    # ── Panel derecho: P(éxito) 0°–180° ──
    _marco_academico(ax2)
    ang_max      = np.pi
    ang_cont     = np.linspace(theta, ang_max, 600)
    p_cont       = np.sin(ang_cont)**2

    ax2.plot(ang_cont * 180 / np.pi, p_cont,
             color=PAL['sol'], linewidth=1.8, zorder=3,
             label=r'$\sin^2\!((2k+1)/2\,\,\theta)$')

    k_disc  = np.arange(0, int(ang_max / (2 * theta)) + 2)
    ang_d   = (2 * k_disc + 1) * theta
    p_d     = np.sin(ang_d)**2
    mask    = ang_d <= ang_max + 1e-9
    ax2.scatter(ang_d[mask] * 180 / np.pi, p_d[mask],
                color=PAL['sol'], s=45, zorder=5, edgecolors='white',
                linewidths=0.8, label='Iteraciones enteras $k$')
    for kv, ad, pd in zip(k_disc[mask], ang_d[mask], p_d[mask]):
        ax2.annotate(f'$k={kv}$\n{pd:.3f}',
                     xy=(ad * 180 / np.pi, pd),
                     xytext=(0, 9), textcoords='offset points',
                     ha='center', fontsize=7.5, color=PAL['sol'])

    # ax2.axvline((2 * R_opt + 1) * theta * 180 / np.pi,
    #             color=PAL['rojo'], linestyle=':', linewidth=1.2,
    #             label=f'$R_{{\\mathrm{{opt}}}} = {R_opt}$')
    # # ax2.axhline(1.0, color=PAL['gris_claro'], lw=0.8, linestyle=':', alpha=0.7)
    # ax2.axhline(M / N, color=PAL['dorado'], lw=1.0, linestyle='--',
    #             alpha=0.8, label=fr'$M/N = {M/N:.4f}$')

    ax2.set_xlim(0, 185)
    ax2.set_ylim(-0.05, 1.20)
    ax2.set_xlabel(r'Ángulo total $(2k+1)/2\,\,\theta$ ($^\circ$)', labelpad=4)
    ax2.set_ylabel(r'$P_{sol}$', labelpad=4)
    ax2.set_title(r'$P_{sol}$ en rango $0^\circ$–$180^\circ$',
                  fontsize=10, pad=6)
    ax2.yaxis.grid(True, linestyle=':', linewidth=0.6, color='#cccccc', alpha=0.8)
    ax2.legend(fontsize=8, loc='upper right')
    _etiqueta_panel(ax2, 'b')

    plt.savefig(ruta('fig4_geometria_grover.png'), dpi=DPI_OUT, bbox_inches='tight')
    plt.close()
    print("  → fig4_geometria_grover.png guardada")


# ──────────────────────────────────────────────
# Figura 5: Comparativa clásico vs cuántico
# ──────────────────────────────────────────────
def graficar_comparativa_clasico_cuantico(N_max_bits=10):
    aplicar_estilo_tfm()
    M_fijo  = 2
    bits    = np.arange(2, N_max_bits + 1)
    N_arr   = 2**bits.astype(float)
    R_clas  = N_arr
    R_grov  = np.array([
        max(1, int(round((np.pi / 4) * np.sqrt(2**n / M_fijo))))
        for n in bits
    ], dtype=float)
    speedup = R_clas / R_grov

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5),
                                   gridspec_kw={'wspace': 0.40})
    fig.suptitle(
        r'Comparativa de complejidad: búsqueda clásica $O(N)$ vs Grover $O(\sqrt{N})$',
        fontsize=11, fontweight='bold', color='#1a1a1a', y=1.02)

    # ── Panel izquierdo: consultas (log) ──
    _marco_academico(ax1)
    ax1.semilogy(bits, R_clas, 's-', color=PAL['rojo'], linewidth=1.8,
                 markersize=6.5, markerfacecolor='white', markeredgewidth=1.8,
                 label=r'Clásico: $O(N)$')
    ax1.semilogy(bits, R_grov, 'o-', color=PAL['sol'], linewidth=1.8,
                 markersize=6.5, markerfacecolor='white', markeredgewidth=1.8,
                 label=r'Grover: $O(\sqrt{N})$')

    # Sudoku 2×2 destacado
    n4, N4 = 4, 16
    R4 = max(1, int(round((np.pi / 4) * np.sqrt(N4 / M_fijo))))
    # ax1.annotate(f'Sudoku $2\\times2$\n$(n=4,\\ R={R4})$',
    #              xy=(4, R4), xytext=(5.2, R4 * 3),
    #              fontsize=8, color=PAL['sol'],
    #              arrowprops=dict(arrowstyle='->', color=PAL['sol'], lw=1.0))
    # ax1.annotate(f'$(n=4,\\ N={N4})$',
    #              xy=(4, N4), xytext=(5.2, N4 * 0.4),
    #              fontsize=8, color=PAL['rojo'],
    #              arrowprops=dict(arrowstyle='->', color=PAL['rojo'], lw=1.0))

    for n_val, rc, rg in zip(bits, R_clas, R_grov):
        ax1.text(n_val, rc * 1.35, f'{int(rc)}', ha='center',
                 fontsize=6.5, color=PAL['rojo'])
        ax1.text(n_val, rg * 0.52, f'{int(rg)}', ha='center',
                 fontsize=6.5, color=PAL['sol'])

    ax1.set_xlabel(r'Número de qubits $n$', labelpad=4)
    ax1.set_ylabel('Consultas al oráculo (escala logarítmica)', labelpad=4)
    ax1.set_title('Número de consultas por algoritmo', fontsize=10, pad=6)
    ax1.set_xticks(bits)
    ax1.set_xticklabels([f'$n={n}$\n$N={2**n}$' for n in bits], fontsize=7.5)
    ax1.legend(fontsize=8.5, loc='upper left')
    ax1.yaxis.grid(True, linestyle=':', linewidth=0.6, color='#cccccc', alpha=0.7,
                   which='both')
    _etiqueta_panel(ax1, 'a')

    # ── Panel derecho: speedup ──
    _marco_academico(ax2)
    ax2.bar(bits, speedup, color=PAL['morado'], edgecolor='white',
            linewidth=0.5, width=0.58, alpha=0.88)
    for n_val, sp in zip(bits, speedup):
        ax2.text(n_val, sp + 0.6, f'$\\times{sp:.1f}$',
                 ha='center', fontsize=8, color=PAL['morado'], fontweight='bold')

    ax2.set_xlabel(r'Número de qubits $n$', labelpad=4)
    ax2.set_ylabel(r'$N\,/\,R_{\mathrm{Grover}}$', labelpad=4)
    ax2.set_title(r'Aceleración cuántica respecto al método clásico', fontsize=10, pad=6)
    ax2.set_xticks(bits)
    ax2.set_xticklabels([f'$n={n}$' for n in bits], fontsize=8.5)
    ax2.set_ylim(0, max(speedup) * 1.22)
    ax2.yaxis.grid(True, linestyle=':', linewidth=0.6, color='#cccccc', alpha=0.7)
    _etiqueta_panel(ax2, 'b')

    plt.savefig(ruta('fig5_comparativa_clasico_cuantico.png'), dpi=DPI_OUT, bbox_inches='tight')
    plt.close()
    print("  → fig5_comparativa_clasico_cuantico.png guardada")


In [57]:


# ============================================================
# EJECUCIÓN DE LA SIMULACIÓN
# ============================================================

if __name__ == '__main__':
    np.random.seed(42)

    # Resolución clásica del problema
    soluciones = encontrar_soluciones()
    print(f"Soluciones del Sudoku 2×2 (M={len(soluciones)}):")
    for s in soluciones:
        vals = [((s >> (3 - i)) & 1) + 1 for i in range(4)]
        print(f"  idx={s:2d} | {format(s,'04b')} | [{vals[0]} {vals[1]}] / [{vals[2]} {vals[3]}]")

    # Resolución cuántica del problema por el algoritmo de Grover
    resultado = algoritmo_grover(soluciones, verbose=True)

    # Verificación del resultado
    idx  = resultado['resultado']
    bits_medidos = [(idx >> (3 - i)) & 1 for i in range(4)]
    vals = [b + 1 for b in bits_medidos]
    print(f"Cuadrícula medida:  | {vals[0]} {vals[1]} |\n"
          f"                    | {vals[2]} {vals[3]} |")
    print(f"¿Solución válida? {'SÍ' if resultado['correcto'] else 'NO'}")

    # Análisis estadístico. 1000 iteraciones
    N_runs = 1000
    medidas_estadisticas = []
    for _ in range(N_runs):
        r = algoritmo_grover(soluciones, verbose=False)
        medidas_estadisticas.append(r['resultado'])
    exitos = sum(1 for m in medidas_estadisticas if m in soluciones)
    print(f"\nAnálisis estadístico ({N_runs} ejecuciones):")
    print(f"  Éxitos: {exitos}/{N_runs}  ({100*exitos/N_runs:.1f}%)")
    print(f"  P(éxito) teórica: {np.sum(resultado['probs_finales'][soluciones]):.4f}")

    # Verificación final del oráculo
    circ_test = CircuitoCuantico(N_TOTAL)
    circ_test.X(FASE_ANC); circ_test.H(FASE_ANC)
    for q in REGISTRO: circ_test.H(q)
    oraculo_sudoku(circ_test)
    sve       = SimuladorVectorEstado()
    estado_or = sve.ejecutar(circ_test)
    psi       = estado_or.reshape(2**N_PRINCIPAL, 2**(N_TOTAL - N_PRINCIPAL))
    marcados  = [x for x in range(16) if np.real(psi[x, 0]) < 0]
    print(f"\nVerificación del oráculo:")
    print(f"  Estados marcados por el oráculo : {marcados}")
    print(f"  Soluciones clásicas             : {soluciones}")
    print(f"  Coincidencia: {marcados == soluciones}")
    print(f"  Ancillas a |0> tras oráculo     : {np.allclose(psi[:, 2:], 0)}")
    print(f"  Norma: {sve.norma():.12f}")

    # Representación
    print("\nGenerando figuras...")
    graficar_distribucion_probabilidad(resultado, soluciones)
    graficar_histograma_estadistico(medidas_estadisticas, soluciones, N_runs)
    graficar_amplificacion_amplitud(resultado, soluciones)
    graficar_rotacion_grover(soluciones, R_max=10)
    graficar_comparativa_clasico_cuantico(N_max_bits=10)



Soluciones del Sudoku 2×2 (M=2):
  idx= 6 | 0110 | [1 2] / [2 1]
  idx= 9 | 1001 | [2 1] / [1 2]

 ALGORITMO DE GROVER — Sudoku 2×2
  Qubits registro : 4  |  Espacio : N = 16
  Soluciones      : M = 2
  Iteraciones     : R = 2  (π/4·√(N/M) = 2.22)
  Complejidad     : clásica O(16), cuántica O(√16≈4.0)
-------------------------------------------------------
  k=0 (inicial): P(solución) = 0.125000  |norma| = 1.0000000000
  k=1 (iter 1): P(solución) = 0.781250  |norma| = 1.0000000000
  k=2 (iter 2): P(solución) = 0.945312  |norma| = 1.0000000000

  Medida: |0110> (índice 6) → ✓ solución

Cuadrícula medida:  | 1 2 |
                    | 2 1 |
¿Solución válida? SÍ

Análisis estadístico (1000 ejecuciones):
  Éxitos: 948/1000  (94.8%)
  P(éxito) teórica: 0.9453

Verificación del oráculo:
  Estados marcados por el oráculo : [6, 9]
  Soluciones clásicas             : [6, 9]
  Coincidencia: True
  Ancillas a |0> tras oráculo     : True
  Norma: 1.000000000000

Generando figuras...
  → fig1_dist

In [ ]:
# ===============================================================================================
# PARTE VI: TRAZA DETALLADA DE UNA ÚNICA EJECUCIÓN DE GROVER
# ===============================================================================================
# Registra el vector de estado completo (11 qubits) en cada punto relevante del circuito,
# sin realizar ninguna medida ni colapso. Permite inspeccionar la evolución coherente
# del sistema paso a paso: preparación, y tras cada aplicación de oráculo + difusión.

def trazar_evolucion_grover(soluciones, R=None, verbose=True):
    """
    Ejecuta una única pasada del algoritmo de Grover registrando el vector de estado
    completo tras cada etapa lógica del circuito, sin medir en ningún momento.

    Parámetros
    ----------
    soluciones : list[int]
        Índices del registro principal que son solución del Sudoku.
    R : int o None
        Número de iteraciones de Grover. Si es None, se usa el óptimo teórico
        R = round((pi/4) * sqrt(N/M)).
    verbose : bool
        Si True, imprime un resumen de cada paso por consola.

    Devuelve
    --------
    traza : list[dict]
        Lista ordenada de snapshots. Cada snapshot es un diccionario con:
          'etapa'        : str, descripción de la etapa ('preparación', 'oráculo k', 'difusión k')
          'iteracion'    : int, número de iteración de Grover (0 para la preparación)
          'estado'       : np.ndarray complejo de tamaño 2**N_TOTAL (vector de estado completo)
          'probs_marg'   : np.ndarray de tamaño 2**N_PRINCIPAL (probabilidad marginal del registro)
          'p_solucion'   : float, probabilidad total acumulada en estados solución
          'norma'        : float, norma del vector de estado (control de consistencia, debe ser 1)
    """
    N = 2**N_PRINCIPAL
    M = len(soluciones)
    if R is None:
        R = max(1, int(round((np.pi / 4) * np.sqrt(N / M))))

    backend = SimuladorVectorEstado()
    backend.n = N_TOTAL
    backend.estado = np.zeros(2**N_TOTAL, dtype=complex)
    backend.estado[0] = 1.0   # |00...0>

    traza = []

    def _registrar(etiqueta, k):
        """Toma una instantánea del estado actual SIN colapsarlo."""
        probs_marg = prob_marginal_registro(backend.estado)
        p_sol = float(np.sum(probs_marg[soluciones]))
        snapshot = {
            'etapa'      : etiqueta,
            'iteracion'  : k,
            'estado'     : backend.estado.copy(),   # copia defensiva: no es una referencia
            'probs_marg' : probs_marg,
            'p_solucion' : p_sol,
            'norma'      : backend.norma(),
        }
        traza.append(snapshot)
        if verbose:
            print(f"  [{etiqueta:>14s}]  P(solución) = {p_sol:.6f}   "
                  f"|psi| = {snapshot['norma']:.10f}")

    if verbose:
        print(f"\n{'='*60}")
        print(f" TRAZA DE EVOLUCIÓN — Grover sobre Sudoku 2×2  (R = {R} iteraciones)")
        print(f"{'='*60}")

    # --- Preparación: ancilla de fase en |-> y superposición uniforme del registro ---
    prep = CircuitoCuantico(N_TOTAL)
    prep.X(FASE_ANC); prep.H(FASE_ANC)
    for q in REGISTRO: prep.H(q)
    for instr in prep.instrucciones:
        backend._aplicar(instr)
    _registrar('preparación', 0)

    # --- Iteraciones de Grover: oráculo seguido de difusión, registrando cada uno ---
    for k in range(1, R + 1):
        circ_oraculo = CircuitoCuantico(N_TOTAL)
        oraculo_sudoku(circ_oraculo)
        for instr in circ_oraculo.instrucciones:
            backend._aplicar(instr)
        _registrar(f'oráculo k={k}', k)

        circ_difusion = CircuitoCuantico(N_TOTAL)
        operador_difusion(circ_difusion)
        for instr in circ_difusion.instrucciones:
            backend._aplicar(instr)
        _registrar(f'difusión k={k}', k)

    if verbose:
        print(f"{'='*60}\n")

    return traza


def resumen_traza(traza, soluciones, n_amplitudes=4):
    """
    Imprime una tabla resumen legible de la traza generada por trazar_evolucion_grover,
    mostrando las n_amplitudes amplitudes de mayor módulo del registro principal
    (trazando sobre ancillas) en cada etapa, sin colapsar el estado en ningún momento.
    """
    print(f"\n{'Etapa':<16s}{'P(sol.)':>10s}{'Norma':>14s}   Amplitudes dominantes (registro, marginal)")
    print("-" * 90)
    for snap in traza:
        probs = snap['probs_marg']
        idx_top = np.argsort(probs)[::-1][:n_amplitudes]
        detalle = ", ".join(
            f"|{format(i,'04b')}⟩:{probs[i]:.4f}{'*' if i in soluciones else ''}"
            for i in idx_top
        )
        print(f"{snap['etapa']:<16s}{snap['p_solucion']:>10.4f}{snap['norma']:>14.10f}   {detalle}")
    print("-" * 90)
    print("  (* indica estado solución del Sudoku)")


# --- Ejecución de la traza ---
traza_grover = trazar_evolucion_grover(soluciones, R=None, verbose=True)
resumen_traza(traza_grover, soluciones, n_amplitudes=4)
